In [2]:
import pandas as pd
import numpy as np

# ============================================================
# STEP 1: Import Dataset
# ============================================================

file_path = "Cleaned_Retention_Cohort_Churn.xlsx"
df = pd.read_excel(file_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)


# ============================================================
# STEP 2: Understand Dataset
# ============================================================

print("\nFirst 5 Rows:")
print(df.head())

print("\nDataset Information:")
print(df.info())

print("\nDescriptive Statistics:")
print(df.describe())


# ============================================================
# STEP 3: Check Data Quality
# ============================================================

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())


# ============================================================
# STEP 4: Basic Business Metrics
# ============================================================

total_users = df["user_id"].nunique()

supply_users = df.loc[
    df["marketplace_side"] == "Supply", "user_id"
].nunique()

demand_users = df.loc[
    df["marketplace_side"] == "Demand", "user_id"
].nunique()

churned_users = df.loc[
    df["churn_flag"] == 1, "user_id"
].nunique()

retained_users = total_users - churned_users

churn_rate = (churned_users / total_users) * 100
retention_rate = (retained_users / total_users) * 100

print("\nBusiness Metrics")
print("----------------")
print("Total Users:", total_users)
print("Supply Users:", supply_users)
print("Demand Users:", demand_users)
print("Retained Users:", retained_users)
print("Churned Users:", churned_users)
print("Retention Rate:", round(retention_rate, 2), "%")
print("Churn Rate:", round(churn_rate, 2), "%")


# ============================================================
# STEP 5: Marketplace Side Analysis
# ============================================================

side_analysis = df.groupby("marketplace_side").agg(
    users=("user_id", "nunique"),
    avg_sessions=("week1_sessions", "mean"),
    avg_searches=("week1_searches", "mean"),
    avg_listings=("week1_listings", "mean"),
    avg_bookings=("week1_bookings", "mean"),
    avg_messages=("week1_messages", "mean"),
    churned_users=("churn_flag", "sum")
).reset_index()

side_analysis["churn_rate"] = (
    side_analysis["churned_users"]
    / side_analysis["users"]
    * 100
)

print("\nMarketplace Side Analysis:")
print(side_analysis)


# ============================================================
# STEP 6: Week-by-Week Retention
# ============================================================

week1_retention = df["active_week_1"].mean() * 100
week2_retention = df["active_week_2"].mean() * 100
week3_retention = df["active_week_3"].mean() * 100
week4_retention = df["active_week_4"].mean() * 100

weekly_retention = pd.DataFrame({
    "week": ["Week 1", "Week 2", "Week 3", "Week 4"],
    "retention_rate": [
        week1_retention,
        week2_retention,
        week3_retention,
        week4_retention
    ]
})

print("\nWeekly Retention:")
print(weekly_retention)


# ============================================================
# STEP 7: Retention by Marketplace Side
# ============================================================

side_retention = df.groupby("marketplace_side").agg(
    week1=("active_week_1", "mean"),
    week2=("active_week_2", "mean"),
    week3=("active_week_3", "mean"),
    week4=("active_week_4", "mean")
).reset_index()

for col in ["week1", "week2", "week3", "week4"]:
    side_retention[col] = side_retention[col] * 100

print("\nRetention by Marketplace Side:")
print(side_retention)


# ============================================================
# STEP 8: Cohort Retention Analysis
# ============================================================

cohort_retention = df.groupby("cohort_month").agg(
    users=("user_id", "nunique"),
    week1_retention=("active_week_1", "mean"),
    week2_retention=("active_week_2", "mean"),
    week3_retention=("active_week_3", "mean"),
    week4_retention=("active_week_4", "mean")
).reset_index()

for col in [
    "week1_retention",
    "week2_retention",
    "week3_retention",
    "week4_retention"
]:
    cohort_retention[col] = cohort_retention[col] * 100

print("\nCohort Retention:")
print(cohort_retention)


# ============================================================
# STEP 9: Cohort Retention by Marketplace Side
# ============================================================

cohort_side = df.groupby(
    ["cohort_month", "marketplace_side"]
).agg(
    users=("user_id", "nunique"),
    week1=("active_week_1", "mean"),
    week2=("active_week_2", "mean"),
    week3=("active_week_3", "mean"),
    week4=("active_week_4", "mean")
).reset_index()

for col in ["week1", "week2", "week3", "week4"]:
    cohort_side[col] = cohort_side[col] * 100

print("\nCohort Retention by Side:")
print(cohort_side)


# ============================================================
# STEP 10: Churn Analysis
# ============================================================

churn_analysis = df.groupby("marketplace_side").agg(
    users=("user_id", "nunique"),
    churned_users=("churn_flag", "sum")
).reset_index()

churn_analysis["churn_rate"] = (
    churn_analysis["churned_users"]
    / churn_analysis["users"]
    * 100
)

print("\nChurn Analysis:")
print(churn_analysis)


# ============================================================
# STEP 11: Week-One Behaviour Comparison
# ============================================================

behavior_comparison = df.groupby("churn_flag").agg(
    users=("user_id", "nunique"),
    avg_sessions=("week1_sessions", "mean"),
    avg_searches=("week1_searches", "mean"),
    avg_listings=("week1_listings", "mean"),
    avg_bookings=("week1_bookings", "mean"),
    avg_messages=("week1_messages", "mean")
).reset_index()

behavior_comparison["status"] = behavior_comparison[
    "churn_flag"
].map({
    0: "Retained",
    1: "Churned"
})

print("\nWeek-One Behaviour Comparison:")
print(behavior_comparison)


# ============================================================
# STEP 12: Engagement Analysis
# ============================================================

engagement_analysis = df.groupby("churn_flag").agg(
    users=("user_id", "nunique"),
    avg_engagement=("week1_engagement", "mean")
).reset_index()

engagement_analysis["status"] = engagement_analysis[
    "churn_flag"
].map({
    0: "Retained",
    1: "Churned"
})

print("\nEngagement Analysis:")
print(engagement_analysis)


# ============================================================
# STEP 13: Booking Behaviour
# ============================================================

booking_analysis = df.groupby("churn_flag").agg(
    users=("user_id", "nunique"),
    avg_bookings=("week1_bookings", "mean"),
    avg_messages=("week1_messages", "mean")
).reset_index()

booking_analysis["status"] = booking_analysis[
    "churn_flag"
].map({
    0: "Retained",
    1: "Churned"
})

print("\nBooking Behaviour:")
print(booking_analysis)


# ============================================================
# STEP 14: Activity Depth Analysis
# ============================================================

activity_analysis = df.groupby("last_active_week").agg(
    users=("user_id", "nunique"),
    avg_sessions=("week1_sessions", "mean"),
    avg_searches=("week1_searches", "mean"),
    avg_bookings=("week1_bookings", "mean"),
    avg_messages=("week1_messages", "mean")
).reset_index()

print("\nActivity Depth Analysis:")
print(activity_analysis)


# ============================================================
# STEP 15: Retention and Engagement Relationship
# ============================================================

engagement_buckets = pd.qcut(
    df["week1_engagement"],
    q=4,
    duplicates="drop"
)

engagement_retention = df.groupby(
    engagement_buckets,
    observed=True
).agg(
    users=("user_id", "nunique"),
    retention_rate=("churn_flag", lambda x: (1 - x.mean()) * 100),
    churn_rate=("churn_flag", "mean")
).reset_index()

engagement_retention["churn_rate"] = (
    engagement_retention["churn_rate"] * 100
)

print("\nEngagement vs Retention:")
print(engagement_retention)


# ============================================================
# STEP 16: Week-One Behaviour Summary
# ============================================================

behavior_summary = df.groupby("churn_flag").agg(
    sessions=("week1_sessions", "mean"),
    searches=("week1_searches", "mean"),
    listings=("week1_listings", "mean"),
    bookings=("week1_bookings", "mean"),
    messages=("week1_messages", "mean"),
    engagement=("week1_engagement", "mean")
).reset_index()

behavior_summary["status"] = behavior_summary[
    "churn_flag"
].map({
    0: "Retained",
    1: "Churned"
})

print("\nBehaviour Summary:")
print(behavior_summary)


# ============================================================
# STEP 17: Key Business Insights
# ============================================================

print("\nKEY BUSINESS INSIGHTS")
print("=====================")

print(
    f"1. Total users analyzed: {total_users:,}"
)

print(
    f"2. Overall retention rate: {retention_rate:.2f}%"
)

print(
    f"3. Overall churn rate: {churn_rate:.2f}%"
)

print(
    f"4. Week 1 retention: {week1_retention:.2f}%"
)

print(
    f"5. Week 2 retention: {week2_retention:.2f}%"
)

print(
    f"6. Week 3 retention: {week3_retention:.2f}%"
)

print(
    f"7. Week 4 retention: {week4_retention:.2f}%"
)

print(
    "8. Week-one behaviour was compared between retained and churned users."
)

print(
    "9. Cohort retention was evaluated across multiple signup cohorts."
)

print(
    "10. Retention was also evaluated separately for both marketplace sides."
)


# ============================================================
# STEP 18: Save Analysis Results
# ============================================================

with pd.ExcelWriter(
    "Python Analysis Result.xlsx",
    engine="openpyxl"
) as writer:

    side_analysis.to_excel(
        writer,
        sheet_name="Side Analysis",
        index=False
    )

    weekly_retention.to_excel(
        writer,
        sheet_name="Weekly Retention",
        index=False
    )

    side_retention.to_excel(
        writer,
        sheet_name="Side Retention",
        index=False
    )

    cohort_retention.to_excel(
        writer,
        sheet_name="Cohort Retention",
        index=False
    )

    cohort_side.to_excel(
        writer,
        sheet_name="Cohort Side",
        index=False
    )

    churn_analysis.to_excel(
        writer,
        sheet_name="Churn Analysis",
        index=False
    )

    behavior_comparison.to_excel(
        writer,
        sheet_name="Behavior Comparison",
        index=False
    )

    engagement_analysis.to_excel(
        writer,
        sheet_name="Engagement Analysis",
        index=False
    )

    engagement_retention.to_excel(
        writer,
        sheet_name="Engagement Retention",
        index=False
    )

print("\nPython Analysis Result.xlsx created successfully.")

Dataset loaded successfully.
Shape: (1500, 16)

First 5 Rows:
  user_id marketplace_side signup_date  week1_sessions  week1_searches  \
0  U10001           Supply  2025-02-17               7               4   
1  U10002           Demand  2025-06-05              12               4   
2  U10003           Demand  2025-04-27               6               4   
3  U10004           Demand  2025-06-03              10               3   
4  U10005           Supply  2025-06-02               7               1   

   week1_listings  week1_bookings  week1_messages  active_week_1  \
0               2               0               0              1   
1               0               1               2              1   
2               0               2               5              1   
3               0               0               2              1   
4               1               0               5              1   

   active_week_2  active_week_3  active_week_4  last_active_week  churn_flag  \
0   